# Verify the canonical 1-arcmin global grid (`Globe_flood_area_*.grb`)

Goal: prove that the ECMWF `Globe_flood_area_YYYYMM.grb` files on
`s3://atlantis/` define the canonical reference grid we should align VIIRS
(and any other) harmonised outputs to, and confirm that `Reprojector` with
`snap_to_global_grid=True` (the default) produces pixel centres that match
that grid exactly.

Reference spec (lat/lon both as pixel **centres**):
```
Ni = 21600 ;  Nj = 10800 ;
lat = 89.99166..., 89.975, 89.95833..., ..., -89.99166...
lon = -179.99166..., -179.975, ..., 179.99166...
```

Note: the GRIB file stores longitudes in the **0–360° convention**, so its
`longitudeOfFirstGridPointInDegrees` reads as `180.00833...°` — the same
pixel as `-179.99166...°` in our `±180°` convention. Coverage is identical;
only the labelling differs.

## Prereqs
- The `ecaws` alias (or `aws --endpoint-url https://object-store.os-api.cci1.ecmwf.int`).
- The `eccodes` Python package + bundled binary lib `eccodeslib` (in this environment).
- ~6 GB free for one month, or ~200 MB to inspect a single message.

## 1. Inspect one GRIB message without downloading the full file

Each message is a global 1-arcmin field (~188 MiB on the wire), so to use
`eccodes` we need at least one full message. We ask S3 for the first 200 MiB.

In [ ]:
import subprocess
from pathlib import Path

# NOTE: `aws s3 cp` does not accept --range; the lower-level
# `aws s3api get-object` does. ~200 MiB covers one full message
# (each `Globe_flood_area_*.grb` message holds a 21600×10800 field).
sample = Path("/tmp/flood_head.grb")
if not sample.exists() or sample.stat().st_size < 200 * 2**20:
    sample.unlink(missing_ok=True)
    subprocess.run(
        [
            "aws", "--endpoint-url", "https://object-store.os-api.cci1.ecmwf.int",
            "s3api", "get-object",
            "--bucket", "atlantis",
            "--key", "Globe_flood_area_202208.grb",
            "--range", "bytes=0-209715199",   # 200 MiB - 1
            str(sample),
        ],
        check=True,
        stdout=subprocess.DEVNULL,   # suppress the JSON response metadata
    )
print(sample, f"{sample.stat().st_size / 2**20:.1f} MiB")

In [ ]:
# Inspect the first GRIB message via the eccodes Python API (no CLI binaries
# needed). Every message in `Globe_flood_area_*.grb` shares the same
# regular_ll grid definition, so message 0 is enough to characterise it.
import eccodes

keys = [
    "shortName",
    "typeOfLevel",
    "dataDate",
    "gridType",
    "Ni",
    "Nj",
    "iDirectionIncrementInDegrees",
    "jDirectionIncrementInDegrees",
    "latitudeOfFirstGridPointInDegrees",
    "latitudeOfLastGridPointInDegrees",
    "longitudeOfFirstGridPointInDegrees",
    "longitudeOfLastGridPointInDegrees",
    "scanningMode",
]

header: dict = {}
with open(sample, "rb") as f:
    gid = eccodes.codes_grib_new_from_file(f)
    try:
        for k in keys:
            try:
                header[k] = eccodes.codes_get(gid, k)
            except eccodes.CodesInternalError:
                pass
    finally:
        eccodes.codes_release(gid)

for k, v in header.items():
    print(f"{k:40s} {v}")

In [ ]:
# Assert the grid matches the canonical 1-arcmin global spec.
import numpy as np

ni        = header["Ni"]
nj        = header["Nj"]
di        = header["iDirectionIncrementInDegrees"]
dj        = header["jDirectionIncrementInDegrees"]
lat0      = header["latitudeOfFirstGridPointInDegrees"]
lat1      = header["latitudeOfLastGridPointInDegrees"]
lon0_grib = header["longitudeOfFirstGridPointInDegrees"]   # 0..360 convention

# Map [0, 360) -> [-180, 180) for comparison with our reprojector.
lon0 = ((lon0_grib + 180.0) % 360.0) - 180.0

print(f"shape          : lat={nj}, lon={ni}")
print(f"spacing        : di={di}, dj={dj}   (1/60 = {1/60:.10f})")
print(f"first lat      : {lat0}    (canonical:  {90.0 - 0.5/60:.6f})")
print(f"last  lat      : {lat1}   (canonical: {-90.0 + 0.5/60:.6f})")
print(f"first lon GRIB : {lon0_grib}   (0..360 convention)")
print(f"first lon ±180 : {lon0}      (canonical: {-180.0 + 0.5/60:.6f})")

# eccodes returns these rounded to ~4 decimals, so use a loose tolerance.
assert (ni, nj) == (21600, 10800), "grid size does not match the 1-arcmin global spec"
assert np.isclose(di, 1/60, atol=1e-4) and np.isclose(dj, 1/60, atol=1e-4), "spacing != 1 arcmin"
assert np.isclose(lat0,  90.0 - 0.5/60, atol=1e-3), "northern pixel centre off canonical position"
assert np.isclose(lat1, -90.0 + 0.5/60, atol=1e-3), "southern pixel centre off canonical position"
assert np.isclose(lon0, -180.0 + 0.5/60, atol=1e-3), "western  pixel centre off canonical position"
print("\nOK — file is on the canonical 1-arcmin global grid (-180, +90, 1/60).")

## 2. (Optional) Open the full file with xarray

This requires the **full** file (~6 GB). Section 1 already proves the grid;
do this step only if you want to actually slice an AOI from the global field.

```bash
mkdir -p data/reference
ecaws s3 cp s3://atlantis/Globe_flood_area_202208.grb data/reference/
```

In [ ]:
from pathlib import Path

full = Path("data/reference/Globe_flood_area_202208.grb")
if not full.exists():
    print(f"Skipping: {full} not found. Run the `ecaws s3 cp` command above first.")
else:
    import xarray as xr

    ds = xr.open_dataset(
        full,
        engine="cfgrib",
        backend_kwargs={"indexpath": ""},   # do not write a .idx next to a 6 GB file
    )
    print(ds)
    print("latitude  first/last :", ds.latitude.values[[0, -1]])
    print("longitude first/last :", ds.longitude.values[[0, -1]])
    print("dlat                 :", float(ds.latitude.values[0] - ds.latitude.values[1]))
    print("dlon                 :", float(ds.longitude.values[1] - ds.longitude.values[0]))

## 3. Confirm our Reprojector now aligns to that grid

We do not need the GRIB file for this — only the **grid definition** it
implies. The check below is purely numeric.

In [ ]:
import numpy as np

from atlantis.harmoniser.reprojector import Reprojector

RES = 1.0 / 60.0
LON0, LAT0 = -180.0, 90.0   # western / northern edges of the global grid

# Canonical pixel centres for the whole globe (±180 convention).
canonical_lon = LON0 + (np.arange(21600) + 0.5) * RES   # -179.9917 .. 179.9917
canonical_lat = LAT0 - (np.arange(10800) + 0.5) * RES   #   89.9917 .. -89.9917
print("global grid:", canonical_lat[0], canonical_lat[-1], canonical_lon[0], canonical_lon[-1])

# West Africa AOI from our latest fetch — intentionally off-grid.
west, south, east, north = -0.86, 8.26, 1.99, 11.73

r = Reprojector(target_resolution=RES, snap_to_global_grid=True)
ws, ss, es, ns = r._snap_bounds_to_global_grid(west, south, east, north)
print("raw   bbox:", (west, south, east, north))
print("snapped   :", (ws, ss, es, ns))

# Indices into the global grid for the snapped window.
i0 = int(round((ws - LON0) / RES))
i1 = int(round((es - LON0) / RES))     # exclusive
j0 = int(round((LAT0 - ns) / RES))     # top row index (north-most)
j1 = int(round((LAT0 - ss) / RES))     # exclusive
print(f"window indices: lon[{i0}:{i1}], lat[{j0}:{j1}]")

# The snapped window's pixel centres MUST equal canonical[i0:i1] / [j0:j1].
window_lon = ws + (np.arange(i1 - i0) + 0.5) * RES
window_lat = ns - (np.arange(j1 - j0) + 0.5) * RES
np.testing.assert_allclose(window_lon, canonical_lon[i0:i1], atol=1e-12)
np.testing.assert_allclose(window_lat, canonical_lat[j0:j1], atol=1e-12)
print("\nOK — snapped AOI window is a bit-for-bit subset of the global grid.")

In [ ]:
import numpy as np
import rioxarray  # noqa: F401
import xarray as xr
from rasterio.transform import from_bounds

from atlantis.harmoniser import Harmoniser
from atlantis.config import HarmoniseConfig

# Tiny synthetic AOI on an arbitrary (off-grid) input.
w, n = -0.853, 11.728
res_in = 0.004
H = W = 80
data = np.random.default_rng(0).uniform(0, 1, (H, W)).astype("float32")
tf = from_bounds(w, n - H * res_in, w + W * res_in, n, W, H)
ds = xr.Dataset(
    {"flood_fraction": xr.DataArray(data, dims=["y", "x"])},
    coords={
        "x": w + (np.arange(W) + 0.5) * res_in,
        "y": n - (np.arange(H) + 0.5) * res_in,
    },
).rio.write_crs("EPSG:4326").rio.write_transform(tf)

ds_h = Harmoniser(HarmoniseConfig()).harmonise(ds)
print(ds_h)

x = ds_h["x"].values
y = ds_h["y"].values
RES = 1.0 / 60.0
kx = (x - (-180.0)) / RES - 0.5
ky = (90.0 - y) / RES - 0.5
print("max |kx-round| =", np.max(np.abs(kx - np.round(kx))))
print("max |ky-round| =", np.max(np.abs(ky - np.round(ky))))
assert np.allclose(kx, np.round(kx), atol=1e-9)
assert np.allclose(ky, np.round(ky), atol=1e-9)
print("OK — harmonised output is on the canonical 1-arcmin global grid.")